## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

In [7]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

while not Path("data/bubbles").exists():
    os.chdir("..")

BUBBLES_DIR = Path("data/bubbles")
VECTOR_DIR = Path("assets/vectorstores")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [8]:
MY_BUBBLE_FILE = "pro_european.jsonl" 

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(r"C:\Users\KlaudiaPoka\OneDrive - BMW Techworks Romania\Desktop\Personal\AIE\echochamber-project-team-2\data\bubbles\pro_european.jsonl", lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: pro_european
Texte: 50


,id,agent,text
0,yt_6_Hc2S02Duw_UgwTw7_YpNZEUGkYDb94AaABAg,Pro-european,"Nu are Ce cauta pe teritoriulRomaniei, indifer..."
1,yt_yEuctxNb4O0_UgxTCkwcP96Sb5_Rpht4AaABAg,Pro-european,Tipul care și-a dat demisia în noiembrie la cu...
2,yt_Fcmw2yFzz8I_UgxuNvJZTDz3pFm5Ijh4AaABAg,Pro-european,Deci exemplu asta cu Parisul cred ca e cea mai...
3,yt_Gk7qe_F1KWE_Ugw6Diry83RDiew6P4V4AaABAg,Pro-european,Ce sunt cu torți trolli ăștia din comentarii? ...
4,yt_5AGO7BTxl_U_UgyL5qOzwKkbeAFQUih4AaABAg,Pro-european,❤❤ NICUȘOR DAN președinte ♥️ MULȚUMIM UE Schen...


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [9]:
texts = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
Nu are Ce cauta pe teritoriulRomaniei, indiferent ca are s-au nu are , ca sint s-au ca nu sint defensive, s-au offensive- nu au acordul poporului


## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [10]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.92it/s]

Număr texte: 50
Dimensiune embeddings: (50, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?
- Câți vectori au fost generați?
- Ce înseamnă a doua valoare din `embeddings.shape`?

- Bula mea are 50 de texte.
- Au fost generați 50 de vectori.
- A doua valoare din embeddings.shape reprezintă dimensiunea fiecărui vector embedding, adică numărul de caracteristici numerice care descriu fiecare text.

In [ ]:
# TODO student:
# Bula mea are     de texte.
# Au fost generați     de vectori.
# A doua valoare din embeddings.shape reprezintă 

## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [11]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: assets\vectorstores\pro_european
Vectori în index: 50


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [ ]:
# TODO student:
# index.faiss există: da
# index.pkl există: da
# index.ntotal este egal cu numărul de texte: 50

## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În următorul continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [12]:
# Text nou introdus în aplicație

input_text = "CCR a decis anularea alegerilor după suspiciuni privind influențe externe."

In [15]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    ["Romania trebuie sa respecte statul de drept si sa ramana alaturi de UE si NATO."],
    normalize_embeddings=True
).astype("float32")

In [ ]:
# query_vector

In [16]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.573
Text: Unde este stegul Uniunii Europene si steagul NATO din imaginea de la Cotroceni?!...eu pt astea două steaguri împreună cu al României am votat 😠...să va fie ruşine România nu e nimeni si nimic fară 🇪🇺 & NATO ...

Rezultat 2
Scor: 0.571
Text: Visul Romaniei de la 1848 a fost sa faca politica Europeana. Sa fie inclusa in Europa si sa fie Europa. Avem acum acest lucru! Ne-am îndeplinit visul iar asta duce la o bunastare fantastica (Romania este cea mai prospera din istorie!) Si exista unii trepanati da ne spuna ca UE nu e nimic. Va dati seama!? Nu zic ca nu mai avem treaba. Mai avem enorm de mult. Dar avem si posibilitatea sa criticam guvernul. Ceea ce ex-EU nu permite. Acolo te “aliniezi” cu interesul national!

Rezultat 3
Scor: 0.524
Text: @Robert Turcescu Oficial: întrebare pentru dnul Călin Georgescu: ce soluție recomandă dânsul pentru ieșirea României din criza în care se află, și întoarcerea la statul de drept și la Constituție? Ce părere are despre iniț

### TODO
Schimbă `input_text` cu o afirmație potrivită pentru agentul tău.
Rulează căutarea.
Notează:
- câte rezultate din 5 sunt relevante;
- dacă textele recuperate exprimă vocea agentului;
- dacă ai observat un text slab care ar trebui eliminat.

- 5 din 5 rezultate sunt relevante.
Toate textele ating teme centrale: UE/NATO, stat de drept, justiție, implicare civică sau susținerea unei direcții occidentale.
- Da, textele exprimă foarte bine vocea agentului.
Se observă clar profilul: pro‑european, pro‑instituțional, orientat spre democrație, justiție funcțională și integrare occidentală.
- Text mai slab:
Rezultatul 4 (cel foarte scurt: „❤❤ NICUȘOR DAN...”) este mai puțin valoros analitic — exprimă susținere, dar nu conține argumente sau idei.